In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cobra
from cobra.core.configuration import Configuration
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import flux_variability_analysis
from tqdm import tqdm

In [12]:
M_xanthus = read_sbml_model("../M_xanthus_model_V4.xml")
M_xanthus

Name,myxo_model
Memory address,7cfe0183c6b0
Number of metabolites,1224
Number of reactions,1339
Number of genes,1201
Number of groups,0
Objective expression,1.0*OF_BIOMASS - 1.0*OF_BIOMASS_reverse_80d2e
Compartments,"c, e"


In [13]:
M_xanthus.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
Fe3_e,EX_Fe3_e,1.065,0,0.00%
alaala_e,EX_alaala_e,507.1,6,22.22%
ca2_e,EX_ca2_e,0.355,0,0.00%
cl_e,EX_cl_e,0.355,0,0.00%
co2_e,EX_co2_e,0.9311,1,0.01%
cobalt2_e,EX_cobalt2_e,0.355,0,0.00%
cu2_e,EX_cu2_e,0.355,0,0.00%
glu_L_e,EX_glu_L_e,198.6,5,7.25%
hdca_e,EX_hdca_e,137.5,16,16.06%
his_L_e,EX_his_L_e,16.97,6,0.74%


In [14]:
iMAT_res = pd.read_csv(
    "/home/mickael/github/M_xanthus-E_coli-Predation/results/Model_V4_iMAT/Ec1/epsilon_1.0_quantiles_40_70.tsv",
    sep="\t",
)
# iMAT_res = pd.read_csv("/home/mickael/github/M_xanthus-E_coli-Predation/results/iMat/reactionData_classification_name.csv", sep=";", index_col="Unnamed: 0")
iMAT_res

,reaction_id,flux_value,classification,y_f,y_r
0,rxn02201_c,0.000000,low,1.0,0.0
1,rxn00351_c,1.084850,high,1.0,0.0
2,rxn07431_c,1.000000,high,1.0,0.0
3,rxn00836_c,-1.000000,high,0.0,1.0
4,rxn01094_c,0.000000,low,1.0,0.0
...,...,...,...,...,...
1334,rxn30548_c,0.000000,NaN,NaN,NaN
1335,rxn09888_c,1.333333,NaN,NaN,NaN
1336,rxn06033_c,-1.000000,NaN,NaN,NaN
1337,EX_hdca_e,-152.180850,NaN,NaN,NaN


In [15]:
iMAT_res_filter = iMAT_res[
    iMAT_res.flux_value != 0
]  # take all the flux different from 0 (so active one)
# iMAT_res_filter = iMAT_res[iMAT_res.flux != 0]
iMAT_res_filter

,reaction_id,flux_value,classification,y_f,y_r
1,rxn00351_c,1.084850,high,1.0,0.0
2,rxn07431_c,1.000000,high,1.0,0.0
3,rxn00836_c,-1.000000,high,0.0,1.0
7,rxn00364_c,7.330270,moderate,NaN,NaN
9,rxn03408_c,1.701158,NaN,NaN,NaN
...,...,...,...,...,...
1332,rxn00899_c,-1.000000,high,0.0,1.0
1335,rxn09888_c,1.333333,NaN,NaN,NaN
1336,rxn06033_c,-1.000000,NaN,NaN,NaN
1337,EX_hdca_e,-152.180850,NaN,NaN,NaN


In [16]:
# # Create dictionary to convert name to id
# Dico_reaction = {}
# for i in M_xanthus.reactions._dict:
#     Dico_reaction[M_xanthus.reactions.get_by_id(i).name] = i
# print(Dico_reaction)

# # create a list of reaction id that should have flux
# active_list = []
# for i in iMAT_res_filter["reaction_id"]:
#     active_list.append(Dico_reaction[i])
# print(active_list)

Check reaction bounds and change the bounds (0.0, 0.0) looking at ModelSeed

In [17]:
# create a list of reaction id that should have flux
active_list = []
for i in iMAT_res_filter["reaction_id"]:
    active_list.append(i)
print(active_list)

['rxn00351_c', 'rxn07431_c', 'rxn00836_c', 'rxn00364_c', 'rxn03408_c', 'rxn00646_c', 'rxn01673_c', 'rxn02342_c', 'rxn05156_c', 'rxn03239_c', 'rxn05457_c', 'rxn00199_c', 'rxn01358_c', 'rxn00172_c', 'rxn02168_c', 'rxn00800_c', 'rxn08333_c', 'rxn01705_c', 'rxn00192_c', 'rxn02898_c', 'rxn10336_c', 'rxn02504_c', 'rxn00262_c', 'rxn05040_c', 'rxn02988_c', 'rxn04792_c', 'rxn00165_c', 'rxn01117_c', 'rxn11268_c', 'rxn00710_c', 'rxn02305_c', 'rxn00910_c', 'rxn01129_c', 'rxn01297_c', 'rxn08796_c', 'rxn08941_c', 'rxn03181_c', 'rxn02331_c', 'rxn06493_c', 'rxn00337_c', 'rxn03248_c', 'rxn08127_c', 'rxn00416_c', 'rxn02376_c', 'rxn12512_c', 'rxn01739_c', 'rxn02679_c', 'rxn01974_c', 'rxn08971_c', 'rxn00060_c', 'rxn00148_c', 'rxn00100_c', 'rxn05560_c', 'rxn08976_c', 'rxn00781_c', 'rxn05298_c', 'rxn02264_c', 'rxn09202_c', 'rxn01451_c', 'rxn00097_c', 'rxn02212_c', 'rxn00695_c', 'rxn10481_c', 'rxn00692_c', 'rxn01492_c', 'rxn00802_c', 'rxn08440_c', 'rxn01825_c', 'rxn02720_c', 'rxn00178_c', 'rxn01362_c', 'rxn0

Shut down all others reactions

In [18]:
for i in M_xanthus.reactions:  # Shut down all the non active reaction
    if i.id not in active_list:
        i.bounds = [0, 0]
list_blocked = cobra.flux_analysis.find_blocked_reactions(M_xanthus)

M_xanthus.optimize()  # Didn't growth

,fluxes,reduced_costs
rxn02201_c,0.000000,8.196139e-01
rxn00351_c,0.000000,-2.633463e-17
rxn07431_c,0.000000,-0.000000e+00
rxn00836_c,-1.537418,1.604619e-17
rxn01094_c,0.000000,-4.006342e-02
...,...,...
rxn30548_c,0.000000,-4.006342e-02
rxn09888_c,0.339588,-6.053869e-19
rxn06033_c,0.000000,-0.000000e+00
EX_hdca_e,-150.763850,0.000000e+00


In [20]:
# cobra.flux_analysis.gapfill(M_xanthus, universal=M_xanthus_ref, exchange_reactions= True, demand_reactions=True, )

Force to activate specific reaction

In [22]:
# Add constraint that force the fluxe to pass there / Infeasable
y = M_xanthus.problem.Variable("y_RXN", type="binary")  # add a binary constraint
M_xanthus.add_cons_vars(y)

for i in active_list:
    have_flux = M_xanthus.problem.Constraint(
        M_xanthus.reactions.get_by_id(i).flux_expression
        - 1
        + 1000
        * (
            1 - y
        ),  # - 1 [threshold] + 1000 [big number] * (1 - y) [if 1 = big number] [if 0 = - big number]
        lb=0,
        ub=0,
    )  # force the reaction to have at least 1 (or -1?) flux
    M_xanthus.add_cons_vars(have_flux)

M_xanthus.solver.update()

In [23]:
M_xanthus.optimize()

/home/mickael/miniconda3/envs/Micka_Predation/lib/python3.12/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


<Solution infeasible at 0x7cfe7d5da3f0>

## **New Data**

python3 IntegrationPackage/main.py weighted_iMAT -f 'data/Raw/WT_vs_4preys.txt' -g 'GeneNames' -i 'W1' -m 'M_xanthus_model_V3_hdca_iMAT.xml' -o results/ -d "quantile"

In [24]:
DataTable = pd.read_csv(
    "/home/mickael/github/M_xanthus-E_coli-Predation/data/Raw/WT_vs_4preys.txt",
    sep="\t",
)
DataTable

,GeneNames,Ec1,Ec2,Ec3,B1,B2,B3,C1,C2,C3,...,W3,K1,K2,K3,WC1,WC2,WC3,KC1,KC2,KC3
0,WP_002614080.1,13032,13136,12917,18050,22828,16816,42888,32404,20170,...,60053,35638,38908,37314,15344,15530,27018,17185,13886,9121
1,WP_002614803.1,6817,5982,6246,15018,16718,18070,29227,34305,9713,...,69546,28536,31303,34021,19022,19870,35902,12388,22264,17843
2,WP_002633201.1,275,195,239,521,550,406,1140,727,386,...,2409,838,832,531,1110,995,1063,1841,1027,766
3,WP_002633598.1,15516,14792,13306,28206,30917,22916,64389,58243,23106,...,62136,27039,28864,34478,7962,9148,24438,4169,7524,7193
4,WP_002633601.1,4095,2680,1969,8170,8806,14771,20987,18394,9361,...,17038,13309,11109,20035,3338,2187,11373,3164,2406,4588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7184,WP_141276995.1,31,36,108,26,31,49,93,69,45,...,76,29,39,42,29,26,39,23,19,33
7185,WP_141277062.1,11,13,46,11,17,4,43,46,17,...,79,10,15,13,14,33,26,17,19,31
7186,WP_143049088.1,1332,1512,1834,2258,2340,1791,4704,5134,1659,...,13345,3235,3431,2941,4105,5714,3841,9265,5144,6080
7187,WP_143049089.1,46,26,305,79,90,72,154,115,44,...,196,89,63,71,102,60,67,100,47,93


In [25]:
DicoTable = pd.read_csv(
    "/home/mickael/github/M_xanthus-E_coli-Predation/data/RNA_seq_DE_result/WT_vs_Ecol_ratio1_3.csv",
    sep=";",
)
DicoTable

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Gene_name,GeneID,orgdb_symbol,orgdb_old_MXAN,orgdb_Name
0,WP_002614080.1,"17681,5595400946","1,13148580809734","0,333820387052008","3,38950481152327","0,000700189800953942","0,00401749202195971",MXAN_RS25275,41362483.0,MXAN_RS25275,MXAN_5201,30S ribosomal protein S21
1,WP_002614803.1,"13055,8658450867","-0,279785484378841","0,341134653438939","-0,820161427630867","0,41212408777437","0,55634844891773",MXAN_RS16100,41360674.0,infA,MXAN_3321,translation initiation factor IF-1
2,WP_002633201.1,"520,781350902195","-0,488659406109829","0,310280848578043","-1,57489386905206","0,115280944069584","0,223684953832229",MXAN_RS31645,41363738.0,MXAN_RS31645,MXAN_6531,HPr family phosphocarrier protein
3,WP_002633598.1,"17667,1430239249","1,78674248251241","0,432318538781286","4,1329305181991","3,58167027156953e-05","0,000318057858780743",MXAN_RS16040,41360662.0,MXAN_RS16040,MXAN_3309,50S ribosomal protein L14
4,WP_002633601.1,"4426,40026145906","0,8104626466811","0,545743984534573","1,48506015576569","0,13752788957159","0,253668278115051",MXAN_RS16025,41360659.0,MXAN_RS16025,MXAN_3306,50S ribosomal protein L16
...,...,...,...,...,...,...,...,...,...,...,...,...
6886,WP_141276995.1,"55,5707618478865","2,38043768070863","0,546005542238998","4,35973171801004","1,30221996444244e-05","0,000133398790047742",NaN,NaN,NaN,NaN,NaN
6887,WP_141277062.1,"25,2414912393685","1,38263010290852","0,762769321629563","1,81264513884054","0,0698865710503559","0,155014402646007",NaN,NaN,NaN,NaN,NaN
6888,WP_143049088.1,"2729,23193040998","0,100695081633711","0,293071005956888","0,343585955577344","0,7311576882233","0,819386506675355",MXAN_RS19445,41361331.0,NaN,NaN,NaN
6889,WP_143049089.1,"109,891161452436","2,01788957892024","0,643239069754101","3,1370755816988","0,0017064213158805","0,008435401210712",NaN,NaN,NaN,NaN,NaN


In [26]:
dico_genes = {}
for i in range(len(DicoTable)):
    dico_genes.update({DicoTable.loc[i, "gene"]: DicoTable.loc[i, "orgdb_old_MXAN"]})
dico_genes

{'WP_002614080.1': 'MXAN_5201',
 'WP_002614803.1': 'MXAN_3321',
 'WP_002633201.1': 'MXAN_6531',
 'WP_002633598.1': 'MXAN_3309',
 'WP_002633601.1': 'MXAN_3306',
 'WP_002633602.1': 'MXAN_3305',
 'WP_002633603.1': 'MXAN_3304',
 'WP_002633604.1': 'MXAN_3303',
 'WP_002633606.1': 'MXAN_3301',
 'WP_002633607.1': 'MXAN_3300',
 'WP_002633608.1': 'MXAN_3299',
 'WP_002634092.1': 'MXAN_5125',
 'WP_002634160.1': 'MXAN_5074',
 'WP_002634235.1': 'MXAN_5002',
 'WP_002634367.1': 'MXAN_5592',
 'WP_002634498.1': 'MXAN_5688',
 'WP_002634858.1': 'MXAN_2709',
 'WP_002635080.1': 'MXAN_2913',
 'WP_002635502.1': 'MXAN_1434',
 'WP_002635980.1': 'MXAN_0672',
 'WP_002636238.1': 'MXAN_0403',
 'WP_002636478.1': 'MXAN_4033',
 'WP_002636551.1': 'MXAN_4095',
 'WP_002636698.1': 'MXAN_4636',
 'WP_002636699.1': 'MXAN_4637',
 'WP_002636736.1': 'MXAN_4673',
 'WP_002637061.1': 'MXAN_2448',
 'WP_002637270.1': 'MXAN_1926',
 'WP_002637381.1': 'MXAN_3295',
 'WP_002637840.1': 'MXAN_7512',
 'WP_002638649.1': 'No old locus tag',
 

In [27]:
DataTable = DataTable.replace(to_replace=dico_genes)
DataTable

,GeneNames,Ec1,Ec2,Ec3,B1,B2,B3,C1,C2,C3,...,W3,K1,K2,K3,WC1,WC2,WC3,KC1,KC2,KC3
0,MXAN_5201,13032,13136,12917,18050,22828,16816,42888,32404,20170,...,60053,35638,38908,37314,15344,15530,27018,17185,13886,9121
1,MXAN_3321,6817,5982,6246,15018,16718,18070,29227,34305,9713,...,69546,28536,31303,34021,19022,19870,35902,12388,22264,17843
2,MXAN_6531,275,195,239,521,550,406,1140,727,386,...,2409,838,832,531,1110,995,1063,1841,1027,766
3,MXAN_3309,15516,14792,13306,28206,30917,22916,64389,58243,23106,...,62136,27039,28864,34478,7962,9148,24438,4169,7524,7193
4,MXAN_3306,4095,2680,1969,8170,8806,14771,20987,18394,9361,...,17038,13309,11109,20035,3338,2187,11373,3164,2406,4588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7184,NaN,31,36,108,26,31,49,93,69,45,...,76,29,39,42,29,26,39,23,19,33
7185,NaN,11,13,46,11,17,4,43,46,17,...,79,10,15,13,14,33,26,17,19,31
7186,NaN,1332,1512,1834,2258,2340,1791,4704,5134,1659,...,13345,3235,3431,2941,4105,5714,3841,9265,5144,6080
7187,NaN,46,26,305,79,90,72,154,115,44,...,196,89,63,71,102,60,67,100,47,93


In [28]:
for i in DataTable.GeneNames:
    print(i)

MXAN_5201
MXAN_3321
MXAN_6531
MXAN_3309
MXAN_3306
MXAN_3305
MXAN_3304
MXAN_3303
MXAN_3301
MXAN_3300
MXAN_3299
MXAN_5125
MXAN_5074
MXAN_5002
MXAN_5592
MXAN_5688
MXAN_2709
MXAN_2913
MXAN_1434
MXAN_0672
MXAN_0403
MXAN_4033
MXAN_4095
MXAN_4636
MXAN_4637
MXAN_4673
MXAN_2448
MXAN_1926
MXAN_3295
MXAN_7512
No old locus tag
MXAN_2528
No old locus tag
MXAN_3793
MXAN_3596
MXAN_2978
MXAN_4894
MXAN_3154
No old locus tag
MXAN_7339
MXAN_2457
MXAN_0001
MXAN_0002
MXAN_0003
MXAN_0004
MXAN_0005
MXAN_0008
MXAN_0009
MXAN_0010
MXAN_0012
MXAN_0013
MXAN_0014
MXAN_0015
MXAN_0016
MXAN_0017
MXAN_0019
MXAN_0020
MXAN_0022
MXAN_0024
nan
MXAN_0026
MXAN_0027
MXAN_0029
MXAN_0030
MXAN_0031
MXAN_0032
MXAN_0033
MXAN_0034
MXAN_0035
WP_011550185.1
MXAN_0039
MXAN_0040
MXAN_0041
MXAN_0042
MXAN_0043
MXAN_0044
MXAN_0045
nan
MXAN_0048
MXAN_0049
MXAN_0050
MXAN_0051
MXAN_0052
MXAN_0053
MXAN_0054
nan
MXAN_0056
MXAN_0057
MXAN_0058
MXAN_0059
MXAN_0060
MXAN_0061
MXAN_0063
MXAN_0064
MXAN_0070
MXAN_0071
MXAN_0072
MXAN_0073
MXAN_0074
MX

In [29]:
for i in DataTable.index:
    if type(DataTable.GeneNames[i]) != str:
        DataTable = DataTable.drop(labels=i)
    if "MXAN" not in DataTable.GeneNames[i]:
        DataTable = DataTable.drop(labels=i)

DataTable.GeneNames

KeyError: 59

In [ ]:
# DataTable.to_csv("/home/mickael/github/M_xanthus-E_coli-Predation/data/Raw/WT_vs_4preys_iMAT.csv", sep = ",")